# Считайте датасет из файла train.csv (это данные о выживаемости на Титанике)
Выберите и обоснуйте метрику для измерения качества (accuracy/precision/recall/f1-score/fbeta-score/roc-auc и т.д.). В рамках данного пункта необходимо подобрать наиболее релевантную метрику или набор метрик для вашей задачи, написав краткое обоснование (1-2 предложения)

In [20]:
import pandas as pd
import numpy as np


df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


На Титанике данные несбалансированы (больше погибших, чем выживших). Accuracy может быть обманчива. F1-score учитывает и точность и полноту (recall), что важно, когда оба типа ошибок (ложно положительные и ложно отрицательные) значимы.



# Произведено разбиение датасета на тренировочную/тестовую выборки - 1 балл
Обеспечена воспроизводимость решения: зафиксированы random_state, ноутбук воспроизводится от начала до конца без ошибок - 2 балла



In [32]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 2025

Разбиение на train/test

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y )

# Произведено измерение качества константного предсказания (например, наиболее частотный класс/случайное предсказание)

 Константный бейслайн

In [36]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
f1_dummy = f1_score(y_test, y_pred_dummy)

print(f"F1-score: {f1_dummy:.4f}")

F1-score: 0.0000


# ML-модель + предобработка


In [24]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features].copy()
y = df['Survived']

X['Age'] = X['Age'].fillna(X['Age'].median())

X = pd.get_dummies(X, columns=['Sex'], drop_first=True)

X.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_male
0,3,22.0,1,0,7.2500,True
1,1,38.0,1,0,71.2833,False
2,3,26.0,0,0,7.9250,False
3,1,35.0,1,0,53.1000,False
4,3,35.0,0,0,8.0500,True


In [25]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

pipe.fit(X_train, y_train)
y_pred_ml = pipe.predict(X_test)

# Измерение качества на отложенной выборке

In [31]:
f1_ml = f1_score(y_test, y_pred_ml)

print(f"F1-score: {f1_ml:.4f}")

F1-score: 0.7630
